# HomeSnitch reproduction scaffold — Python 3.12

This notebook reproduces the **traffic-analysis and machine-learning workflow** from:

> TJ O'Connor et al., *HomeSnitch: Behavior Transparency and Control for Smart Home IoT Devices*, ACM WiSec 2019.

The paper classifies smart-home IoT semantic behaviors from encrypted TCP traffic by reconstructing connection-oriented application data unit (ADU) exchanges and calculating 13 header-derived features.

## Reproduction boundary

The public YourThings release provides PCAP files and a device mapping CSV. The HomeSnitch paper does not publish its exact 148-class semantic label manifest, full `adudump` preprocessing configuration, Random Forest hyperparameters, or Ryu/OpenFlow prototype source.

This notebook therefore provides a **transparent, configurable reproduction scaffold**:

- A portable ADU reconstruction approximation using `tshark`.
- The exact 13 features listed in Table 1.
- Timestamp-window semantic labeling.
- Random Forest, kNN, and Gradient Boost evaluation.
- Unknown-behavior rejection and UBMR sweep.
- Required-training-sample experiment.
- DBSCAN cluster discovery.
- Behavior reports.
- Offline policy-impact simulation.
- Optional iPerf-result analysis.

It does not claim byte-for-byte equivalence with the original `adudump` pipeline or live SDN enforcement.

## 1. Dependencies

Install the recommended system parser:

```bash
sudo apt-get install tshark
```

Install Python 3.12 packages:

```bash
python -m pip install numpy pandas scikit-learn matplotlib joblib tqdm pyarrow
```

`tshark` is strongly preferred for the 10–13 GB YourThings PCAP files because it streams fields without loading captures into memory.

In [ ]:
from __future__ import annotations

import heapq
import json
import shutil
import subprocess
import sys
import warnings
from dataclasses import dataclass, field
from datetime import datetime, timezone
from glob import glob
from pathlib import Path
from typing import Dict, Iterator, List, Optional, Sequence, Tuple

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.cluster import DBSCAN
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

print("Python:", sys.version)
print("tshark:", shutil.which("tshark") or "not installed — synthetic mode still works")
print("pandas:", pd.__version__)

## 2. Configuration

Official YourThings data page: <https://yourthings.info/data/>.

Start with the synthetic demo, then download the mapping CSV and **one** April PCAP. Do not start by parsing all captures.

In [ ]:
# ---------------------------
# USER CONFIGURATION
# ---------------------------
RUN_SYNTHETIC_DEMO = True

PCAP_GLOB = "/path/to/yourthings/*.pcap*"
DEVICE_MAPPING_CSV = Path("/path/to/device_mapping.csv")
BEHAVIOR_LABELS_CSV = Path("/path/to/homesnitch_behavior_labels.csv")

OUTPUT_DIR = Path("./homesnitch_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Explicit approximation settings. The HomeSnitch paper does not report these.
FLOW_TIMEOUT_S = 120.0
QUIET_TIME_S = 0.5
MIN_TOTAL_PAYLOAD_BYTES = 1

# Explicit model defaults. The HomeSnitch paper does not report hyperparameters.
RANDOM_SEED = 7
N_SPLITS = 10
RF_ESTIMATORS = 100
UNKNOWN_THRESHOLD = 0.89

# Expensive experiments: keep defaults small while validating the pipeline.
MAX_UNKNOWN_CLASSES = None
MAX_TRAINING_CURVE_CLASSES = 2
TRAINING_CURVE_STEPS = [1, 2, 5, 10, 20, 40, 60, 80, 100]

print("Synthetic demo:", RUN_SYNTHETIC_DEMO)
print("Output directory:", OUTPUT_DIR.resolve())

## 3. HomeSnitch feature list

Each row is one TCP connection-oriented ADU dialogue. A **sequence** is one directional application exchange within that connection.

In [ ]:
FEATURE_COLUMNS = [
    "avg_iot_bytes_per_sequence",
    "avg_server_bytes_per_sequence",
    "aggregate_server_bytes_over_adu",
    "aggregate_iot_bytes_over_adu",
    "min_iot_bytes_in_sequence",
    "min_server_bytes_in_sequence",
    "max_server_bytes_in_sequence",
    "max_iot_bytes_in_sequence",
    "stdev_server_sequence_bytes",
    "stdev_iot_sequence_bytes",
    "server_sequences_per_adu",
    "iot_sequences_per_adu",
    "connection_duration_s",
]

PAPER_FEATURE_IMPORTANCE = pd.Series({
    "avg_iot_bytes_per_sequence": 0.213104,
    "avg_server_bytes_per_sequence": 0.072519,
    "aggregate_server_bytes_over_adu": 0.105775,
    "aggregate_iot_bytes_over_adu": 0.117552,
    "min_iot_bytes_in_sequence": 0.038917,
    "min_server_bytes_in_sequence": 0.038344,
    "max_server_bytes_in_sequence": 0.079063,
    "max_iot_bytes_in_sequence": 0.135909,
    "stdev_server_sequence_bytes": 0.054491,
    "stdev_iot_sequence_bytes": 0.050798,
    "server_sequences_per_adu": 0.013566,
    "iot_sequences_per_adu": 0.016211,
    "connection_duration_s": 0.063750,
}, name="paper_importance").sort_values(ascending=False)

display(PAPER_FEATURE_IMPORTANCE.to_frame())

## 4. Device mapping loader

The public data page supplies a local device mapping CSV. The loader accepts several plausible header names; inspect the normalized mapping before extraction.

In [ ]:
def _pick_column(columns: Sequence[str], aliases: Sequence[str]) -> Optional[str]:
    lowered = {str(col).strip().lower(): col for col in columns}
    for alias in aliases:
        if alias.lower() in lowered:
            return lowered[alias.lower()]
    return None

def load_device_mapping(path: Path) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Device mapping not found: {path}")
    raw = pd.read_csv(path)
    ip_col = _pick_column(raw.columns, ["local_ip", "device_ip", "ip", "ip_address", "address"])
    name_col = _pick_column(raw.columns, ["device_name", "device", "name", "label"])
    vendor_col = _pick_column(raw.columns, ["vendor", "manufacturer", "brand"])
    model_col = _pick_column(raw.columns, ["model", "device_model", "product"])
    if ip_col is None:
        raise ValueError(f"Cannot infer IP column. Columns: {list(raw.columns)}")
    out = pd.DataFrame()
    out["local_ip"] = raw[ip_col].astype(str).str.strip()
    out["device_name"] = raw[name_col].astype(str).str.strip() if name_col else out["local_ip"]
    out["vendor"] = raw[vendor_col].astype(str).str.strip() if vendor_col else out["device_name"]
    out["model"] = raw[model_col].astype(str).str.strip() if model_col else out["device_name"]
    return out.drop_duplicates(subset=["local_ip"]).reset_index(drop=True)

if DEVICE_MAPPING_CSV.exists():
    device_mapping = load_device_mapping(DEVICE_MAPPING_CSV)
    display(device_mapping.head(20))
else:
    device_mapping = pd.DataFrame(columns=["local_ip", "device_name", "vendor", "model"])
    print("Mapping not loaded. Set DEVICE_MAPPING_CSV when switching to real PCAPs.")

## 5. Portable ADU reconstruction approximation

The paper uses `adudump`. The implementation below follows the stated abstraction but is not a drop-in replacement.

Rules:

1. Stream IPv4 TCP packet metadata with `tshark`.
2. Keep flows involving a mapped IoT IP.
3. Deduplicate retransmitted payload ranges approximately with TCP sequence numbers.
4. Group adjacent payload-bearing packets in the same direction into a sequence.
5. Split a same-direction sequence after `QUIET_TIME_S`.
6. Flush a flow on FIN/RST, timeout, or PCAP boundary.
7. Calculate the Table-1 features.

In [ ]:
@dataclass(frozen=True)
class TcpPacket:
    ts: float
    src_ip: str
    dst_ip: str
    src_port: int
    dst_port: int
    seq: int
    payload_len: int
    syn: bool
    ack: bool
    fin: bool
    rst: bool

def _bool(value: str) -> bool:
    return str(value).strip() in {"1", "True", "true", "yes"}

def iter_tshark_tcp_packets(pcap_path: Path) -> Iterator[TcpPacket]:
    tshark = shutil.which("tshark")
    if not tshark:
        raise RuntimeError("Install tshark before parsing PCAP files.")
    fields = [
        "frame.time_epoch", "ip.src", "ip.dst", "tcp.srcport", "tcp.dstport",
        "tcp.seq_raw", "tcp.len", "tcp.flags.syn", "tcp.flags.ack",
        "tcp.flags.fin", "tcp.flags.reset",
    ]
    cmd = [tshark, "-n", "-r", str(pcap_path), "-Y", "tcp && ip", "-T", "fields"]
    for field_name in fields:
        cmd += ["-e", field_name]
    cmd += ["-E", "separator=\t", "-E", "occurrence=f", "-o", "tcp.relative_sequence_numbers:FALSE"]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, bufsize=1)
    assert proc.stdout is not None
    for line in proc.stdout:
        parts = line.rstrip("\n").split("\t")
        if len(parts) != len(fields):
            continue
        try:
            yield TcpPacket(
                ts=float(parts[0]), src_ip=parts[1], dst_ip=parts[2],
                src_port=int(parts[3]), dst_port=int(parts[4]),
                seq=int(parts[5] or 0), payload_len=int(parts[6] or 0),
                syn=_bool(parts[7]), ack=_bool(parts[8]),
                fin=_bool(parts[9]), rst=_bool(parts[10]),
            )
        except ValueError:
            continue
    stderr = proc.stderr.read() if proc.stderr else ""
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"tshark failed for {pcap_path}: {stderr[-1000:]}")

def canonical_flow_key(packet: TcpPacket) -> Tuple[Tuple[str, int], Tuple[str, int]]:
    a, b = sorted([(packet.src_ip, packet.src_port), (packet.dst_ip, packet.dst_port)])
    return a, b

def merge_interval_count_unseen(intervals: List[Tuple[int, int]], start: int, end: int) -> int:
    if end <= start:
        return 0
    unseen = end - start
    new_start, new_end = start, end
    merged = []
    inserted = False
    for old_start, old_end in intervals:
        if old_end < new_start:
            merged.append((old_start, old_end))
        elif new_end < old_start:
            if not inserted:
                merged.append((new_start, new_end))
                inserted = True
            merged.append((old_start, old_end))
        else:
            unseen -= max(0, min(new_end, old_end) - max(new_start, old_start))
            new_start, new_end = min(new_start, old_start), max(new_end, old_end)
    if not inserted:
        merged.append((new_start, new_end))
    intervals[:] = merged
    return max(0, unseen)

def safe_mean(values) -> float:
    return float(np.mean(values)) if len(values) else 0.0

def safe_min(values) -> float:
    return float(np.min(values)) if len(values) else 0.0

def safe_max(values) -> float:
    return float(np.max(values)) if len(values) else 0.0

def safe_std(values) -> float:
    return float(np.std(values, ddof=0)) if len(values) else 0.0

@dataclass
class FlowState:
    key: Tuple[Tuple[str, int], Tuple[str, int]]
    pcap_name: str
    iot_ip: str
    iot_port: int
    peer_ip: str
    peer_port: int
    start_ts: float
    last_ts: float
    last_payload_ts: Optional[float] = None
    current_direction: Optional[str] = None
    current_bytes: int = 0
    iot_sequences: List[int] = field(default_factory=list)
    server_sequences: List[int] = field(default_factory=list)
    signed_pattern: List[int] = field(default_factory=list)
    seen_intervals: Dict[str, List[Tuple[int, int]]] = field(default_factory=lambda: {"iot": [], "server": []})

    def flush_sequence(self):
        if self.current_direction is None or self.current_bytes <= 0:
            return
        if self.current_direction == "iot":
            self.iot_sequences.append(self.current_bytes)
            self.signed_pattern.append(self.current_bytes)
        else:
            self.server_sequences.append(self.current_bytes)
            self.signed_pattern.append(-self.current_bytes)
        self.current_direction, self.current_bytes = None, 0

    def add(self, packet: TcpPacket, quiet_time_s: float):
        self.last_ts = max(self.last_ts, packet.ts)
        if packet.payload_len <= 0:
            return
        direction = "iot" if packet.src_ip == self.iot_ip else "server"
        unseen = merge_interval_count_unseen(
            self.seen_intervals[direction], packet.seq, packet.seq + packet.payload_len
        )
        if unseen <= 0:
            return
        quiet_gap = self.last_payload_ts is not None and packet.ts - self.last_payload_ts > quiet_time_s
        if self.current_direction is None:
            self.current_direction, self.current_bytes = direction, unseen
        elif direction != self.current_direction or quiet_gap:
            self.flush_sequence()
            self.current_direction, self.current_bytes = direction, unseen
        else:
            self.current_bytes += unseen
        self.last_payload_ts = packet.ts

    def finalize(self) -> Optional[Dict[str, object]]:
        self.flush_sequence()
        iot_total, server_total = sum(self.iot_sequences), sum(self.server_sequences)
        if iot_total + server_total < MIN_TOTAL_PAYLOAD_BYTES:
            return None
        return {
            "pcap_name": self.pcap_name,
            "start_ts": self.start_ts, "end_ts": self.last_ts,
            "start_time_utc": datetime.fromtimestamp(self.start_ts, tz=timezone.utc).isoformat(),
            "end_time_utc": datetime.fromtimestamp(self.last_ts, tz=timezone.utc).isoformat(),
            "iot_ip": self.iot_ip, "iot_port": self.iot_port,
            "peer_ip": self.peer_ip, "peer_port": self.peer_port,
            "signed_adu_pattern": json.dumps(self.signed_pattern),
            "avg_iot_bytes_per_sequence": safe_mean(self.iot_sequences),
            "avg_server_bytes_per_sequence": safe_mean(self.server_sequences),
            "aggregate_server_bytes_over_adu": float(server_total),
            "aggregate_iot_bytes_over_adu": float(iot_total),
            "min_iot_bytes_in_sequence": safe_min(self.iot_sequences),
            "min_server_bytes_in_sequence": safe_min(self.server_sequences),
            "max_server_bytes_in_sequence": safe_max(self.server_sequences),
            "max_iot_bytes_in_sequence": safe_max(self.iot_sequences),
            "stdev_server_sequence_bytes": safe_std(self.server_sequences),
            "stdev_iot_sequence_bytes": safe_std(self.iot_sequences),
            "server_sequences_per_adu": float(len(self.server_sequences)),
            "iot_sequences_per_adu": float(len(self.iot_sequences)),
            "connection_duration_s": max(0.0, float(self.last_ts - self.start_ts)),
        }

def choose_iot_endpoint(packet: TcpPacket, iot_ips: set[str]):
    src_iot, dst_iot = packet.src_ip in iot_ips, packet.dst_ip in iot_ips
    if not src_iot and not dst_iot:
        return None
    if src_iot and not dst_iot:
        return packet.src_ip, packet.src_port, packet.dst_ip, packet.dst_port
    if dst_iot and not src_iot:
        return packet.dst_ip, packet.dst_port, packet.src_ip, packet.src_port
    # For IoT-to-IoT LAN flows, prefer the visible initiator.
    return packet.src_ip, packet.src_port, packet.dst_ip, packet.dst_port

def extract_adu_features_from_pcap(
    pcap_path: Path,
    iot_ips: set[str],
    flow_timeout_s: float = FLOW_TIMEOUT_S,
    quiet_time_s: float = QUIET_TIME_S,
) -> pd.DataFrame:
    active = {}
    expiry_heap = []
    rows = []

    def flush(key):
        state = active.pop(key, None)
        if state:
            row = state.finalize()
            if row:
                rows.append(row)

    def expire(now):
        cutoff = now - flow_timeout_s
        while expiry_heap and expiry_heap[0][0] < cutoff:
            old_ts, key = heapq.heappop(expiry_heap)
            state = active.get(key)
            if state and state.last_ts == old_ts:
                flush(key)

    for packet in iter_tshark_tcp_packets(Path(pcap_path)):
        expire(packet.ts)
        endpoint = choose_iot_endpoint(packet, iot_ips)
        if endpoint is None:
            continue
        iot_ip, iot_port, peer_ip, peer_port = endpoint
        key = canonical_flow_key(packet)
        state = active.get(key)
        if state is None or packet.ts - state.last_ts > flow_timeout_s:
            if state is not None:
                flush(key)
            state = FlowState(
                key=key, pcap_name=Path(pcap_path).name,
                iot_ip=iot_ip, iot_port=iot_port, peer_ip=peer_ip, peer_port=peer_port,
                start_ts=packet.ts, last_ts=packet.ts,
            )
            active[key] = state
        state.add(packet, quiet_time_s)
        heapq.heappush(expiry_heap, (state.last_ts, key))
        if packet.fin or packet.rst:
            flush(key)

    for key in list(active):
        flush(key)
    return pd.DataFrame(rows)

def extract_adu_features_from_pcaps(pcap_paths: Sequence[Path], mapping: pd.DataFrame, output_csv: Path) -> pd.DataFrame:
    iot_ips = set(mapping["local_ip"].astype(str))
    output_csv = Path(output_csv)
    frames = []
    for index, pcap_path in enumerate(sorted(map(Path, pcap_paths)), start=1):
        print(f"[{index}/{len(pcap_paths)}] {pcap_path}")
        frame = extract_adu_features_from_pcap(pcap_path, iot_ips)
        print(f"  -> {len(frame):,} ADU rows")
        frames.append(frame)
        if not frame.empty:
            frame.to_csv(output_csv, mode="a", header=not output_csv.exists(), index=False)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

## 6. Extract YourThings PCAPs

This cell is skipped in synthetic mode. For real data, cache extracted ADU features so model iteration does not require re-parsing PCAPs.

In [ ]:
REAL_ADU_CSV = OUTPUT_DIR / "yourthings_adu_features.csv"

if not RUN_SYNTHETIC_DEMO:
    pcap_paths = [Path(p) for p in sorted(glob(PCAP_GLOB))]
    if not pcap_paths:
        raise FileNotFoundError(f"No files matched: {PCAP_GLOB}")
    if device_mapping.empty:
        raise RuntimeError("Load DEVICE_MAPPING_CSV first.")
    if REAL_ADU_CSV.exists():
        adus_unlabeled = pd.read_csv(REAL_ADU_CSV)
        print("Loaded cached rows:", len(adus_unlabeled))
    else:
        adus_unlabeled = extract_adu_features_from_pcaps(pcap_paths, device_mapping, REAL_ADU_CSV)
        print("Saved:", REAL_ADU_CSV)
    display(adus_unlabeled.head())
else:
    adus_unlabeled = pd.DataFrame()
    print("Synthetic mode: PCAP extraction skipped.")

## 7. Timestamp-window semantic labels

The paper's exact 148-class label manifest is not public. Create your own manifest with:

```text
start_time,end_time,device_ip,vendor,model,activity,pcap_name,notes
```

The assignment below labels an ADU when its start timestamp falls inside a matching device window.

In [ ]:
def load_behavior_manifest(path: Path) -> pd.DataFrame:
    manifest = pd.read_csv(path)
    required = {"start_time", "end_time", "device_ip", "vendor", "model", "activity"}
    missing = required - set(manifest.columns)
    if missing:
        raise ValueError(f"Manifest missing: {sorted(missing)}")
    manifest = manifest.copy()
    manifest["start_ts"] = pd.to_datetime(manifest["start_time"], utc=True).astype("int64") / 1e9
    manifest["end_ts"] = pd.to_datetime(manifest["end_time"], utc=True).astype("int64") / 1e9
    manifest["device_ip"] = manifest["device_ip"].astype(str)
    if "pcap_name" not in manifest:
        manifest["pcap_name"] = ""
    manifest["pcap_name"] = manifest["pcap_name"].fillna("").astype(str)
    manifest["behavior_label"] = (
        manifest["vendor"].astype(str) + "|" + manifest["model"].astype(str) + "|" + manifest["activity"].astype(str)
    )
    return manifest

def assign_behavior_labels(adus: pd.DataFrame, manifest: pd.DataFrame) -> pd.DataFrame:
    out = adus.copy()
    for col in ["behavior_label", "vendor", "model", "activity"]:
        out[col] = pd.NA
    out["label_match_count"] = 0
    for _, window in manifest.iterrows():
        mask = (
            (out["iot_ip"].astype(str) == str(window["device_ip"])) &
            (out["start_ts"] >= float(window["start_ts"])) &
            (out["start_ts"] <= float(window["end_ts"]))
        )
        if str(window["pcap_name"]).strip():
            mask &= out["pcap_name"].astype(str) == str(window["pcap_name"])
        out.loc[mask, "label_match_count"] += 1
        first = mask & out["behavior_label"].isna()
        for col in ["behavior_label", "vendor", "model", "activity"]:
            out.loc[first, col] = window[col]
    overlaps = int((out["label_match_count"] > 1).sum())
    if overlaps:
        print(f"Warning: {overlaps:,} rows matched multiple label windows.")
    return out

if not RUN_SYNTHETIC_DEMO:
    manifest = load_behavior_manifest(BEHAVIOR_LABELS_CSV)
    adus_labeled = assign_behavior_labels(adus_unlabeled, manifest)
    display(adus_labeled.head())
else:
    print("Synthetic mode: labels generated below.")

## 8. Synthetic smoke-test dataset

This checks notebook wiring without the raw captures. It is not evidence for the paper's claims.

In [ ]:
def make_synthetic_dataset(n_per_class: int = 120, seed: int = RANDOM_SEED) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    specs = [
        ("Ring", "Doorbell", "Heartbeat", 70, 80, 1, 1, 4.0),
        ("Ring", "Doorbell", "MotionUpload", 2800, 450, 2, 2, 9.0),
        ("Hue", "Bridge", "LightControl", 180, 100, 2, 2, 1.0),
        ("Canary", "Camera", "Heartbeat", 110, 150, 2, 3, 12.0),
        ("Canary", "Camera", "VideoUpload", 9000, 900, 4, 3, 45.0),
        ("TP-Link", "HS110", "FirmwareDownload", 350, 12000, 3, 5, 38.0),
    ]
    start = pd.Timestamp("2018-04-13T00:00:00Z").timestamp()
    rows = []
    for class_index, (vendor, model, activity, iot_scale, server_scale, iot_n, server_n, duration) in enumerate(specs):
        for sample_index in range(n_per_class):
            iot = np.maximum(1, rng.normal(iot_scale, max(3, iot_scale * .1), max(1, int(round(rng.normal(iot_n, .35)))))).astype(int)
            server = np.maximum(1, rng.normal(server_scale, max(3, server_scale * .1), max(1, int(round(rng.normal(server_n, .35)))))).astype(int)
            signed = []
            for i in range(max(len(iot), len(server))):
                if i < len(iot): signed.append(int(iot[i]))
                if i < len(server): signed.append(-int(server[i]))
            ts = start + class_index * 20000 + sample_index * max(2, duration)
            row = {
                "pcap_name": "synthetic-demo.pcap",
                "start_ts": ts, "end_ts": ts + duration,
                "start_time_utc": datetime.fromtimestamp(ts, tz=timezone.utc).isoformat(),
                "end_time_utc": datetime.fromtimestamp(ts + duration, tz=timezone.utc).isoformat(),
                "iot_ip": f"192.168.1.{10 + class_index}", "iot_port": 30000 + sample_index,
                "peer_ip": f"52.0.{class_index}.{10 + sample_index % 7}",
                "peer_port": 1883 if activity == "Heartbeat" else 443,
                "signed_adu_pattern": json.dumps(signed),
                "avg_iot_bytes_per_sequence": safe_mean(iot),
                "avg_server_bytes_per_sequence": safe_mean(server),
                "aggregate_server_bytes_over_adu": float(server.sum()),
                "aggregate_iot_bytes_over_adu": float(iot.sum()),
                "min_iot_bytes_in_sequence": safe_min(iot),
                "min_server_bytes_in_sequence": safe_min(server),
                "max_server_bytes_in_sequence": safe_max(server),
                "max_iot_bytes_in_sequence": safe_max(iot),
                "stdev_server_sequence_bytes": safe_std(server),
                "stdev_iot_sequence_bytes": safe_std(iot),
                "server_sequences_per_adu": float(len(server)),
                "iot_sequences_per_adu": float(len(iot)),
                "connection_duration_s": max(.01, float(rng.normal(duration, max(.1, duration * .08)))),
                "vendor": vendor, "model": model, "activity": activity,
                "behavior_label": f"{vendor}|{model}|{activity}",
                "label_match_count": 1,
            }
            rows.append(row)
    return pd.DataFrame(rows)

if RUN_SYNTHETIC_DEMO:
    adus_labeled = make_synthetic_dataset()

model_df = adus_labeled[adus_labeled["behavior_label"].notna()].copy()
print("Rows:", len(model_df), "Classes:", model_df["behavior_label"].nunique())
display(model_df["behavior_label"].value_counts().rename("rows").to_frame())
display(model_df.head())

## 9. Quality checks

High cross-validation scores can be misleading if duplicate rows, overlapping label windows, or activation-session leakage make train/test separation trivial. For publication-quality work, also evaluate group-based splits by day or activation session.

In [ ]:
print("Missing features:")
display(model_df[FEATURE_COLUMNS].isna().sum().rename("missing").to_frame())
print("Duplicate labeled feature rows:", model_df.duplicated(subset=FEATURE_COLUMNS + ["behavior_label"]).sum())

plt.figure(figsize=(7, 4))
plt.hist(model_df["connection_duration_s"], bins=40)
plt.xlabel("Connection duration (seconds)")
plt.ylabel("ADU rows")
plt.title("Duration distribution")
plt.show()

## 10. Stratified 10-fold model comparison

The paper reports stratified 10-fold accuracy, recall, and F1 for kNN, Gradient Boost, and Random Forest. The implementation below uses macro recall and macro F1.

In [ ]:
def build_models():
    return {
        "k-Nearest Neighbor": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
        "Gradient Boost": GradientBoostingClassifier(random_state=RANDOM_SEED),
        "Random Forest": RandomForestClassifier(n_estimators=RF_ESTIMATORS, random_state=RANDOM_SEED, n_jobs=-1),
    }

def evaluate_models(df: pd.DataFrame, n_splits: int = N_SPLITS) -> pd.DataFrame:
    counts = df["behavior_label"].value_counts()
    eligible = counts[counts >= n_splits].index
    work = df[df["behavior_label"].isin(eligible)]
    X, y = work[FEATURE_COLUMNS].astype(float), work["behavior_label"].astype(str)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    rows = []
    for name, model in build_models().items():
        result = cross_validate(
            model, X, y, cv=cv,
            scoring={"accuracy": "accuracy", "recall": "recall_macro", "f1": "f1_macro"},
            n_jobs=1,
        )
        rows.append({
            "model": name,
            "accuracy_mean": result["test_accuracy"].mean(), "accuracy_std": result["test_accuracy"].std(),
            "recall_macro_mean": result["test_recall"].mean(), "recall_macro_std": result["test_recall"].std(),
            "f1_macro_mean": result["test_f1"].mean(), "f1_macro_std": result["test_f1"].std(),
            "rows": len(work), "classes": work["behavior_label"].nunique(),
        })
    return pd.DataFrame(rows).sort_values("accuracy_mean", ascending=False)

cv_results = evaluate_models(model_df)
display(cv_results)

## 11. Random Forest feature importance

Compare learned rankings against Table 1. A materially different ranking can indicate different ADU boundaries, labels, device mix, or traffic phase.

In [ ]:
X, y = model_df[FEATURE_COLUMNS].astype(float), model_df["behavior_label"].astype(str)
rf_model = RandomForestClassifier(n_estimators=RF_ESTIMATORS, random_state=RANDOM_SEED, n_jobs=-1)
rf_model.fit(X, y)

learned = pd.Series(rf_model.feature_importances_, index=FEATURE_COLUMNS, name="learned_importance")
importance = pd.concat([learned, PAPER_FEATURE_IMPORTANCE], axis=1).sort_values("learned_importance", ascending=False)
display(importance)

plt.figure(figsize=(8, 5))
importance["learned_importance"].sort_values().plot(kind="barh")
plt.xlabel("Importance")
plt.title("Learned Random Forest feature importance")
plt.tight_layout()
plt.show()

## 12. Known-behavior confidence CDF

Figure 5 in the paper plots confidence scores. This cell uses out-of-fold probabilities so a row is not scored by the same model that trained on it.

In [ ]:
def known_oof_scores(df: pd.DataFrame, n_splits: int = N_SPLITS) -> pd.DataFrame:
    counts = df["behavior_label"].value_counts()
    work = df[df["behavior_label"].isin(counts[counts >= n_splits].index)].copy()
    X, y = work[FEATURE_COLUMNS].astype(float), work["behavior_label"].astype(str)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    model = RandomForestClassifier(n_estimators=RF_ESTIMATORS, random_state=RANDOM_SEED, n_jobs=-1)
    proba = cross_val_predict(model, X, y, cv=cv, method="predict_proba", n_jobs=1)
    classes = np.array(sorted(y.unique()))
    out = work[["behavior_label"]].reset_index(drop=True)
    out["predicted_label"] = classes[proba.argmax(axis=1)]
    out["max_probability"] = proba.max(axis=1)
    out["correct"] = out["behavior_label"] == out["predicted_label"]
    return out

known_scores = known_oof_scores(model_df)
print("OOF accuracy:", known_scores["correct"].mean())

scores = np.sort(known_scores["max_probability"].to_numpy())
plt.figure(figsize=(7, 4))
plt.plot(scores, np.arange(1, len(scores) + 1) / len(scores))
plt.xlabel("Maximum predicted probability")
plt.ylabel("CDF")
plt.title("Known-behavior confidence CDF")
plt.grid(True, alpha=.3)
plt.show()

## 13. Unknown-behavior rejection and UBMR

For each behavior class, remove all of its rows from training, train on the remaining classes, and score the held-out rows. A held-out row is a missed unknown behavior when its maximum probability exceeds the acceptance threshold.

In [ ]:
def leave_one_behavior_out_scores(df: pd.DataFrame, max_classes=MAX_UNKNOWN_CLASSES) -> pd.DataFrame:
    labels = sorted(df["behavior_label"].astype(str).unique())
    if max_classes is not None:
        labels = labels[:max_classes]
    rows = []
    for i, held_out in enumerate(labels, start=1):
        train = df[df["behavior_label"].astype(str) != held_out]
        test = df[df["behavior_label"].astype(str) == held_out]
        if train["behavior_label"].nunique() < 2:
            continue
        model = RandomForestClassifier(n_estimators=RF_ESTIMATORS, random_state=RANDOM_SEED, n_jobs=-1)
        model.fit(train[FEATURE_COLUMNS].astype(float), train["behavior_label"].astype(str))
        for score in model.predict_proba(test[FEATURE_COLUMNS].astype(float)).max(axis=1):
            rows.append({"held_out_label": held_out, "max_probability": float(score)})
        print(f"[{i}/{len(labels)}] {held_out}")
    return pd.DataFrame(rows)

def threshold_sweep(known: pd.DataFrame, unknown: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for threshold in np.linspace(0, 1, 101):
        known_accept = known["max_probability"] >= threshold
        unknown_accept = unknown["max_probability"] >= threshold
        rows.append({
            "threshold": threshold,
            "known_accept_rate": known_accept.mean(),
            "known_accepted_correct_rate": (known_accept & known["correct"]).mean(),
            "ubmr": unknown_accept.mean(),
            "unknown_detection_rate": (~unknown_accept).mean(),
        })
    return pd.DataFrame(rows)

unknown_scores = leave_one_behavior_out_scores(model_df)
sweep = threshold_sweep(known_scores, unknown_scores)
op = sweep.iloc[(sweep["threshold"] - UNKNOWN_THRESHOLD).abs().argsort()[:1]]
display(op)

plt.figure(figsize=(7, 5))
plt.plot(sweep["ubmr"], sweep["known_accepted_correct_rate"], marker=".")
plt.scatter(op["ubmr"], op["known_accepted_correct_rate"], s=80)
plt.xlabel("Unknown Behavior Miss Rate (UBMR)")
plt.ylabel("Known accepted-correct rate")
plt.title("Novelty-rejection trade-off")
plt.grid(True, alpha=.3)
plt.show()

## 14. Required training samples

The paper reports an average of 58.82 samples for 90% confidence and 62.28 for 95%. The full experiment is expensive. Use a coarse sweep first; for a closer reproduction set `TRAINING_CURVE_STEPS = list(range(1, 201))` and test every class.

In [ ]:
def training_sample_curve(df: pd.DataFrame, steps=TRAINING_CURVE_STEPS, max_classes=MAX_TRAINING_CURVE_CLASSES):
    rng = np.random.default_rng(RANDOM_SEED)
    labels = sorted(df["behavior_label"].astype(str).unique())
    if max_classes is not None:
        labels = labels[:max_classes]
    rows = []
    for i, label in enumerate(labels, start=1):
        target = df[df["behavior_label"].astype(str) == label].iloc[rng.permutation((df["behavior_label"].astype(str) == label).sum())].reset_index(drop=True)
        other = df[df["behavior_label"].astype(str) != label]
        for n_train in steps:
            if n_train <= 0 or n_train >= len(target):
                continue
            train = pd.concat([other, target.iloc[:n_train]], ignore_index=True)
            remain = target.iloc[n_train:]
            model = RandomForestClassifier(n_estimators=RF_ESTIMATORS, random_state=RANDOM_SEED, n_jobs=-1)
            model.fit(train[FEATURE_COLUMNS].astype(float), train["behavior_label"].astype(str))
            class_index = list(model.classes_).index(label)
            score = model.predict_proba(remain[FEATURE_COLUMNS].astype(float))[:, class_index].mean()
            rows.append({"behavior_label": label, "n_train_samples": n_train, "mean_target_probability": score})
        print(f"[{i}/{len(labels)}] {label}")
    return pd.DataFrame(rows)

training_curve = training_sample_curve(model_df)
display(training_curve.head())

plt.figure(figsize=(8, 5))
for label, group in training_curve.groupby("behavior_label"):
    plt.plot(group["n_train_samples"], group["mean_target_probability"], marker="o", label=label)
plt.axhline(.90, linestyle="--")
plt.axhline(.95, linestyle="--")
plt.xlabel("Training samples added")
plt.ylabel("Mean target-class probability")
plt.title("Required-training-sample experiment")
plt.legend(fontsize=7)
plt.grid(True, alpha=.3)
plt.show()

## 15. DBSCAN cluster discovery

For a new deployment, cluster unlabeled ADUs and manually correlate clusters with observed actions. DBSCAN parameters are not disclosed in the paper, so expose and tune them explicitly.

In [ ]:
def plot_k_distance(df: pd.DataFrame, neighbors=5):
    Xs = StandardScaler().fit_transform(df[FEATURE_COLUMNS].astype(float))
    distances, _ = NearestNeighbors(n_neighbors=neighbors).fit(Xs).kneighbors(Xs)
    plt.figure(figsize=(7, 4))
    plt.plot(np.sort(distances[:, -1]))
    plt.xlabel("Sorted rows")
    plt.ylabel(f"Distance to {neighbors}th neighbor")
    plt.title("DBSCAN k-distance diagnostic")
    plt.grid(True, alpha=.3)
    plt.show()

def cluster_adus(df: pd.DataFrame, eps=1.4, min_samples=5):
    out = df.copy()
    Xs = StandardScaler().fit_transform(out[FEATURE_COLUMNS].astype(float))
    out["dbscan_cluster"] = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(Xs)
    return out

cluster_source = adus_unlabeled if not adus_unlabeled.empty else model_df.drop(columns=["behavior_label"], errors="ignore")
plot_k_distance(cluster_source)
clustered_adus = cluster_adus(cluster_source)
display(clustered_adus["dbscan_cluster"].value_counts().sort_index().rename("rows").to_frame())

## 16. Behavior report

Appendix B combines structural, destination, and timing metadata to help a human assign semantic labels.

In [ ]:
def behavior_report(df: pd.DataFrame, label: str) -> dict:
    subset = df[df["behavior_label"].astype(str) == str(label)].copy()
    if subset.empty:
        raise ValueError(label)
    return {
        "label": label,
        "flows": len(subset),
        "devices": subset["iot_ip"].value_counts().head(5).to_dict(),
        "top_destination_ports": subset["peer_port"].value_counts().head(5).to_dict(),
        "top_destination_addresses": subset["peer_ip"].value_counts().head(5).to_dict(),
        "top_signed_patterns": subset["signed_adu_pattern"].value_counts().head(5).to_dict(),
        "iot_bytes_p75": float(subset["aggregate_iot_bytes_over_adu"].quantile(.75)),
        "server_bytes_p75": float(subset["aggregate_server_bytes_over_adu"].quantile(.75)),
        "duration_median_s": float(subset["connection_duration_s"].median()),
        "duration_mean_s": float(subset["connection_duration_s"].mean()),
        "duration_p75_s": float(subset["connection_duration_s"].quantile(.75)),
        "recent_flows": subset.sort_values("start_ts", ascending=False)[
            ["start_time_utc", "iot_ip", "iot_port", "peer_ip", "peer_port"]
        ].head(3).to_dict(orient="records"),
    }

sample_label = model_df["behavior_label"].value_counts().index[0]
print(json.dumps(behavior_report(model_df, sample_label), indent=2))

## 17. Offline policy-impact simulation

This estimates the flows and bytes affected by a behavior-aware rule. It does not configure OpenFlow.

In [ ]:
def simulate_policy(df: pd.DataFrame, behavior_label: str, action="drop", iot_ip=None, peer_port=None):
    subset = df[df["behavior_label"].astype(str) == str(behavior_label)].copy()
    if iot_ip is not None:
        subset = subset[subset["iot_ip"].astype(str) == str(iot_ip)]
    if peer_port is not None:
        subset = subset[subset["peer_port"].astype(int) == int(peer_port)]
    iot_bytes = subset["aggregate_iot_bytes_over_adu"].sum()
    server_bytes = subset["aggregate_server_bytes_over_adu"].sum()
    return {
        "action": action,
        "behavior_label": behavior_label,
        "matching_flows": int(len(subset)),
        "iot_bytes_affected": float(iot_bytes),
        "server_bytes_affected": float(server_bytes),
        "total_payload_bytes_affected": float(iot_bytes + server_bytes),
    }

policy_label = "Ring|Doorbell|MotionUpload" if "Ring|Doorbell|MotionUpload" in set(model_df["behavior_label"]) else model_df["behavior_label"].iloc[0]
print(json.dumps(simulate_policy(model_df, policy_label), indent=2))

## 18. Optional iPerf analysis for RQ2

PCAPs cannot reproduce SDN performance overhead. Populate `homesnitch_iperf_results.csv` with:

```text
variant,parallel_connections,iteration,throughput_mbps,latency_ms
```

In [ ]:
IPERF_RESULTS_CSV = Path("./homesnitch_iperf_results.csv")

if IPERF_RESULTS_CSV.exists():
    iperf = pd.read_csv(IPERF_RESULTS_CSV)
    summary = iperf.groupby(["variant", "parallel_connections"]).agg(
        throughput_mbps_mean=("throughput_mbps", "mean"),
        throughput_mbps_std=("throughput_mbps", "std"),
        latency_ms_mean=("latency_ms", "mean"),
        latency_ms_std=("latency_ms", "std"),
        runs=("iteration", "count"),
    ).reset_index()
    display(summary)
else:
    print("Optional iPerf CSV not present.")

## 19. Export model, features, and configuration

Preserve intermediate features and parameters. A serious replication should also record PCAP checksums and label-manifest provenance.

In [ ]:
MODEL_PATH = OUTPUT_DIR / "homesnitch_random_forest.joblib"
FEATURES_PATH = OUTPUT_DIR / "homesnitch_labeled_adu_features.csv"
CONFIG_PATH = OUTPUT_DIR / "homesnitch_experiment_config.json"

joblib.dump(rf_model, MODEL_PATH)
model_df.to_csv(FEATURES_PATH, index=False)
CONFIG_PATH.write_text(json.dumps({
    "run_synthetic_demo": RUN_SYNTHETIC_DEMO,
    "pcap_glob": PCAP_GLOB,
    "device_mapping_csv": str(DEVICE_MAPPING_CSV),
    "behavior_labels_csv": str(BEHAVIOR_LABELS_CSV),
    "flow_timeout_s": FLOW_TIMEOUT_S,
    "quiet_time_s": QUIET_TIME_S,
    "min_total_payload_bytes": MIN_TOTAL_PAYLOAD_BYTES,
    "random_seed": RANDOM_SEED,
    "n_splits": N_SPLITS,
    "rf_estimators": RF_ESTIMATORS,
    "unknown_threshold": UNKNOWN_THRESHOLD,
    "features": FEATURE_COLUMNS,
}, indent=2), encoding="utf-8")

print("Saved:", MODEL_PATH)
print("Saved:", FEATURES_PATH)
print("Saved:", CONFIG_PATH)

## 20. Next steps

1. Run synthetic mode end-to-end.
2. Parse one April YourThings PCAP.
3. Inspect ADU patterns manually.
4. Build a small timestamped ground-truth manifest.
5. Add group-based validation by day or activation session.
6. Compare this extractor against original `adudump` output if obtainable.
7. Tune quiet time, flow timeout, DBSCAN parameters, Random Forest settings, and novelty threshold.
8. Treat clusters as unlabeled until externally validated.
9. Implement OpenFlow only after offline rules are checked for collateral blocking and delayed buffered uploads.